Devemos começar por 

In [2]:
import numpy as np
import pandas as pd
from pathlib import Path

In [ ]:
path = Path("../../train.csv")
text = path.read_text(encoding='utf-8')

lines = text.splitlines()

print(f"Nº linhas: {len(lines)}")
print(f"Primeira linha: {lines[0]}")


Nº linhas: 2673
Primeira linha: medical_specialty;description;sample_name;transcription;keywords


In [9]:
suspicious = []

for number, line in enumerate(lines[1:], start=2):
    if line.count(";") != 4:
        suspicious.append((number, line.count(";"), line[:10]))

print("Linhas suspeitas:", len(suspicious))
print(suspicious[:10])

Linhas suspeitas: 13
[(161, 1, 'Cardiovasc'), (163, 6, 'Psychiatry'), (714, 1, 'General Me'), (947, 1, 'Neurosurge'), (961, 6, 'General Me'), (1088, 1, 'Obstetrics'), (1096, 6, 'General Me'), (2089, 1, 'Surgery;"T'), (2100, 6, 'Neurology;'), (2329, 1, 'Surgery;"A')]


In [12]:
odd_quotes = []

for number, line in enumerate(lines[1:], start=2):
    if line.count('"') % 2 != 0:
        odd_quotes.append((number, line.count('"'), line[:10]))

print("Linhas com número ímpar de aspas:", len(odd_quotes))
print(odd_quotes)

Linhas com número ímpar de aspas: 14
[(161, 1, 'Cardiovasc'), (163, 7, 'Psychiatry'), (714, 1, 'General Me'), (726, 13, 'Obstetrics'), (947, 1, 'Neurosurge'), (961, 1, 'General Me'), (1088, 1, 'Obstetrics'), (1096, 7, 'General Me'), (2089, 1, 'Surgery;"T'), (2100, 1, 'Neurology;'), (2329, 1, 'Surgery;"A'), (2337, 1, 'Surgery; D'), (2394, 1, 'Surgery;"L'), (2395, 23, 'Radiology;')]


In [ ]:
header, body = lines[0], lines[1:]
LABELS =[]
for number, line in enumerate (body, start=2):
    first = line.split(";",1)[0]
    if first not in LABELS and len(first) < 100:
        LABELS.append(first)
print(LABELS)


['Cardiovascular-Pulmonary', 'Gastroenterology', 'Psychiatry-Psychology', 'Dermatology', 'General Medicine', 'Ophthalmology', 'Neurosurgery', 'Orthopedic', 'Surgery', 'Radiology', 'Obstetrics-Gynecology', 'Neurology']
12


In [19]:
new_record_lines = []
continuation_lines =[]

for line_number, line in enumerate(body,start=2):
    first = line.split(";",1)[0].strip()

    if first in LABELS:
        new_record_lines.append((line_number, line))
    else:
        continuation_lines.append((line_number, line))

print(f"NEW: {len(new_record_lines)}")
print(f"CONTINUATION: {len(continuation_lines)}")


NEW: 2669
CONTINUATION: 3


In [20]:
print("Exemplos de linhas que começam registos:")
for line_number, line in new_record_lines[:3]:
    print(line_number, line[:120])

print("\nExemplos de linhas que não começam registos:")
for line_number, line in continuation_lines[:5]:
    print(line_number, line[:120])

Exemplos de linhas que começam registos:
2 Cardiovascular-Pulmonary; Chest pain, possible syncopal spells.  She has been having multiple cardiovascular complaints 
3 Cardiovascular-Pulmonary;The patient is 14 months old, comes in with a chief complaint of difficulty breathing.; Difficu
4 Cardiovascular-Pulmonary; Fiberoptic bronchoscopy, diagnostic.  Hemoptysis and history of lung cancer.  Tumor occluding 

Exemplos de linhas que não começam registos:
962 d is open about may be roughly a centimeter in the inside of the flap.  He was seen by his medical primary care physicia
2101 "ypothyroidism secondary to hyperthyroidism and irradiation, possible lumbar stenosis.  He reports he experienced a sing
2338 ted and this was used to aspirate approximately 30 cc of serosanguineous fluid from the cystic structure.  Next, the nee


In [22]:
for line_number, line in continuation_lines[:20]:
    print(
        line_number,
        "separadores =", line.count(";"),
        "aspas =", line.count('"'),
        repr(line[:40])
    )

962 separadores = 4 aspas = 0 'd is open about may be roughly a centime'
2101 separadores = 4 aspas = 16 '"ypothyroidism secondary to hyperthyroid'
2338 separadores = 4 aspas = 0 'ted and this was used to aspirate approx'


In [36]:
def classify_line(line):
    first_field = line.split(";", 1)[0].strip()

    if line.count(";") == 1 and line.count('"') % 2 == 1:
        return "fragment"

    if first_field in LABELS:
        return "new_record"


    if (first_field not in LABELS) and line.endswith(";;;;"):
        return "continuation"

    return "unknown"

In [38]:
from collections import Counter

line_types = Counter(classify_line(line) for line in body)
line_types

Counter({'new_record': 2662, 'fragment': 7, 'continuation': 3})

In [72]:
records = []
current_record = None
continuation_lines = []

for line_number, line in enumerate(body, start=2):
    kind = classify_line(line)

    if kind == "fragment":
        continue

    if kind == "new_record":
        records.append(line)

    elif kind == "continuation":
        print(f"cont line: {len(records) - 1}")
        continuation_lines.append((line_number, line))
        records[-1] += line

cont line: 956
cont line: 2092
cont line: 2327


In [73]:
for x in range(len(continuation_lines)):
    print(continuation_lines[x][1][-100:])


s to keep the wound clean with the moist dressing right now.  He may shower several times a day.;;;;
4.  We will follow the lumbar stenosis for the time being.  No further workup is planned.""""""";;;;
en from the operating room in stable condition and should be observed overnight in the hospital.;;;;


In [74]:
records[2327][-100:]

'en from the operating room in stable condition and should be observed overnight in the hospital.;;;;'

In [75]:
semis = pd.Series([line.count(";") for line in records])
semis.value_counts().sort_index()

4     2656
6        3
10       3
Name: count, dtype: int64